<div style="margin:auto;width:80%;">
Here is the complete architectural layout connecting your local Mac workspace, Docker runtime, Snowflake data warehouse, and GitHub repository.

```text
+-----------------------------------------------------------------------------------+
|                                LOCAL MAC HOST                                     |
|                                                                                   |
|  [ Workspace & Code ]                         [ Version Control ]                 |
|  - olist_docker/ (Notebooks, dbt, Dagster) -> git push -> GitHub Remote (origin)  |
|  - .env (Excluded via .gitignore)                                                 |
|  - ~/.ssh/snowflake_rsa_key.p8 (RSA Private Key)                                  |
+---------------------------------- | ----------------------------------------------+
                                    | Volume Mounts & Port Forwarding
                                    v
+-----------------------------------------------------------------------------------+
|                            DOCKER ENGINE ("THE FACTORY")                          |
|                                                                                   |
|   +-----------------------------------+   +-----------------------------------+   |
|   | olist_dagster (Port 3000)         |   | olist_jupyter (Port 8888)         |   |
|   |                                   |   |                                   |   |
|   | - Dagster Orchestrator UI         |   | - JupyterLab Workspace            |   |
|   | - Runs dbt-snowflake models       |   | - Pandas / PyArrow / Seaborn      |   |
|   | - Executes pipeline assets        |   | - Runs exploratory notebooks      |   |
|   +-----------------+-----------------+   +-----------------+-----------------+   |
+---------------------|---------------------------------------|---------------------+
                      | dbt Run / Materialize                 | SQL Query / Fetch
                      v                                       v
+-----------------------------------------------------------------------------------+
|                            SNOWFLAKE DATA WAREHOUSE                               |
|                                                                                   |
|  - Database: ANALYTICS                                                            |
|  - Schemas: STAGING -> MARTS_LOGISTICS                                            |
|  - Tables/Views: stg_olist__orders, fct_deliveries, ml_order_review_training      |
+-----------------------------------------------------------------------------------+

```

---

**1. Code & Local Filesystem (Mac Host $\leftrightarrow$ Docker)**

* **Two-Way Mirror (`./:/app`):** Any edit you make in VS Code or JupyterLab immediately Syncs across your local disk and container filesystem.
* **Security Isolation:** Your `.env` secrets file and `.ssh` RSA private key live on your Mac. The key is mounted into Docker as a read-only volume (`:ro`), allowing authentication with Snowflake without baking secrets into the Docker image.

**2. Containerized Workloads (Inside Docker)**

* **`olist_dagster` Container:** Houses the Dagster webserver and dbt engine. When triggered at `localhost:3000`, it connects to Snowflake via key-pair authentication to execute raw data loads and dbt model transformations.
* **`olist_jupyter` Container:** Houses your analytics environment at `localhost:8888`. It uses `snowflake-connector-python` and `pyarrow` to pull structured data into Pandas DataFrames and plot Seaborn/Matplotlib charts.

**3. Version Control & Governance (Git $\rightarrow$ GitHub)**

* **Public/Private Repo:** Pushes your infrastructure blueprints (`Dockerfile`, `docker-compose.yml`), dbt SQL models, Dagster assets, and Python notebooks to GitHub.
* **`.gitignore` Shield:** Blocks local virtual environments (`venv_dbt_docker/`), runtime caches, `.env` credentials, and SSH keys from ever being exposed to remote repositories.
</div>

<div style="margin:auto;width:80%;">
Here is your 5-level developer iteration loop for updating code, SQL models, and pipelines safely without breaking production state.

1. **Level 1: Local Code & SQL Development:** Edit files directly in VS Code.
Edit your dbt models (`.sql`), Dagster assets (`.py`), or requirements directly on your Mac. Because of your Docker bind mount (`./:/app`), every file edit syncs into the running containers instantly without requiring a Docker rebuild.


2. **Level 2: Model Testing via dbt CLI:** Test transformations in Snowflake.
Run and test specific models inside the container to verify SQL syntax and data quality assertions:

```bash
# Run a specific modified dbt model
docker exec -it olist_dagster dbt run --select fct_deliveries --project-dir dbt_olist_github/dbt_olistgithub

# Run data tests against Snowflake
docker exec -it olist_dagster dbt test --project-dir dbt_olist_github/dbt_olistgithub

```


3. **Level 3: Pipeline Orchestration Sync:** Validate assets in Dagster UI.
Open **http://localhost:3000** in your browser:

* If you edited `dagster_olist/olist_assets.py`, click **Reload Definitions** in the top-right corner to refresh the asset graph.
* Click **Materialize** on updated assets to verify end-to-end execution and dependency flow.


4. **Level 4: Exploratory Verification:** Verify data in JupyterLab.
Open **http://localhost:8888** (or use your local `venv_dbt_docker` kernel in VS Code):

* Run pandas/Snowflake queries to inspect target table schemas and values.
* Update Seaborn plots to confirm downstream analytical insights reflect your SQL changes.


5. **Level 5: Git Version Control & Deployment:** Commit and push to GitHub.
Once verified, stage and push your clean updates to remote:

```bash
# 1. Verify secrets (.env, keys) remain untracked
git status

# 2. Stage, commit, and push
git add .
git commit -m "refactor: update fct_deliveries SLA calculation logic"
git push origin main

```
</div>